In [15]:
import numpy as np
import pandas as pd
import pathlib

## Data Source
Data used for this project are the flat text files for the AACT database snapshot of **December 2025**. Each text file corresponds to a table in the schema.

Snapshot used will be found under "Monthly Archives".
[Link here](https://aact.ctti-clinicaltrials.org/downloads/snapshots?type=flatfiles)

In [16]:
raw = pathlib.Path('../data/raw')

## Labels and Exclusions
A trial's overall status will serve as the label for the model. Only trials in the database with an overall status of either completed or terminated will be included, meaning the target is a binary outcome. This also means any trials which were ongoing at the time of the database snapshot will be excluded.

In [17]:
studies = pd.read_csv(raw/'studies.txt', sep='|', low_memory=False)
studies_cols = [
    'nct_id', 'biospec_retention', 'is_ppsd', 'phase',
    'is_unapproved_device', 'is_fda_regulated_device', 'is_fda_regulated_drug',
    'number_of_groups', 'number_of_arms', 'overall_status'
]

studies = studies[studies_cols]
# to ensure label is a binary outcome
studies = studies[studies['overall_status'].isin(['COMPLETED', 'TERMINATED'])]

In [18]:
calc_vals = pd.read_csv(raw/'calculated_values.txt', sep='|')
calc_vals_cols = [
    'nct_id', 'number_of_facilities', 'has_us_facility',
    'minimum_age_num', 'minimum_age_unit', 'maximum_age_num', 'maximum_age_unit',
    'number_of_primary_outcomes_to_measure', 'number_of_secondary_outcomes_to_measure'
]

calc_vals = calc_vals[calc_vals_cols]

# scaling all age values to years
scalers = {
    'year':1.0, 'month':1.0/12.0, 'week':7.0/365.25,
    'day':1.0/365.25, 'hour':1.0/365.25/24.0, 'minute':1.0/365.25/24.0/60.0,
    np.nan:np.nan
}

calc_vals['max_age_years'] = calc_vals['maximum_age_unit'].apply(lambda x: scalers[x]) * calc_vals['maximum_age_num']
calc_vals['min_age_years'] = calc_vals['minimum_age_unit'].apply(lambda x: scalers[x]) * calc_vals['minimum_age_num']

# dropping original age num columns and units
# since after scaling they become redundant
calc_vals.drop(
    columns=[
        'maximum_age_unit', 'minimum_age_unit',
        'maximum_age_num', 'minimum_age_num'
    ],
    inplace=True
)

In [19]:
designs = pd.read_csv(raw/'designs.txt', sep='|')
design_cols = [
    'nct_id', 'intervention_model', 'observational_model', 'primary_purpose'
]

designs = designs[design_cols]

# intervention_model and observational_model are never both not nan
# so will coalesce them into a model column
# also will add a study_type column to indicate intervention vs. observational
designs['study_type'] = [
    'intervention' if i else 'observational' if o else np.nan
    for i, o in zip(designs['intervention_model'].notna(), designs['observational_model'].notna())
]
designs['model'] = [
    i if pd.notna(i) else o
    for i, o in zip(designs['intervention_model'], designs['observational_model'])
]

designs.drop(columns=['intervention_model', 'observational_model'], inplace=True)

In [20]:
# only feature used from this table is total number of countries involved for a trial
countries = pd.read_csv(raw/'countries.txt', sep='|')
countries = pd.DataFrame(countries.groupby('nct_id')['name'].nunique()).reset_index()
countries.columns = ['nct_id', 'num_countries']

In [21]:
sponsors = pd.read_csv(raw/'sponsors.txt', sep='|')
sponsors = sponsors[sponsors['lead_or_collaborator'] == 'lead'][['nct_id', 'agency_class']]

In [22]:
# merging of all tables, left joins to avoid losing rows
final_table = studies
for table in [calc_vals, designs, countries, sponsors]:
    final_table = pd.merge(final_table, table, how='left', on='nct_id')
final_table['terminated'] = final_table['overall_status'] == 'TERMINATED'
final_table.drop(columns='overall_status', inplace=True)

In [23]:
# final row count
print(final_table.shape[0], "total samples")

# confirming each row corresponds to unique trial (nct_id)
print("trials unique?", final_table['nct_id'].nunique() == final_table.shape[0])

# low prevalence should be taken into consideration when testing models
print("prevalence:", final_table['terminated'].mean())

338720 total samples
trials unique? True
prevalence: 0.0956748937175248


In [24]:
final_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338720 entries, 0 to 338719
Data columns (total 21 columns):
 #   Column                                   Non-Null Count   Dtype  
---  ------                                   --------------   -----  
 0   nct_id                                   338720 non-null  object 
 1   biospec_retention                        12553 non-null   object 
 2   is_ppsd                                  6 non-null       object 
 3   phase                                    142408 non-null  object 
 4   is_unapproved_device                     2323 non-null    object 
 5   is_fda_regulated_device                  159455 non-null  object 
 6   is_fda_regulated_drug                    159463 non-null  object 
 7   number_of_groups                         46366 non-null   float64
 8   number_of_arms                           248592 non-null  float64
 9   number_of_facilities                     338720 non-null  int64  
 10  has_us_facility                 

In [25]:
# may be better to drop some highly sparse features
final_table.isna().mean().sort_values(ascending=False)

is_ppsd                                    0.999982
is_unapproved_device                       0.993142
biospec_retention                          0.962940
number_of_groups                           0.863114
phase                                      0.579570
is_fda_regulated_device                    0.529242
is_fda_regulated_drug                      0.529219
max_age_years                              0.464688
number_of_secondary_outcomes_to_measure    0.278330
number_of_arms                             0.266084
primary_purpose                            0.225750
has_us_facility                            0.073943
min_age_years                              0.065464
num_countries                              0.062624
number_of_primary_outcomes_to_measure      0.036963
study_type                                 0.030299
model                                      0.030299
agency_class                               0.000000
nct_id                                     0.000000
number_of_fa

In [26]:
# drop features here
features_to_drop = ['is_ppsd', 'biospec_retention', 'is_unapproved_device']
for feature in features_to_drop:
    try:
        final_table.drop(feature, axis=1, inplace=True)
    except KeyError:
        pass

## Feature Set v1
- Kept features:
    - phase
    - is_fda_regulated_device
    - is_fda_regulated_drug
    - number_of_groups
    - number_of_arms
    - number_of_facilities
    - has_us_facility
    - number_of_primary_outcomes_to_measure
    - number_of_secondary_outcomes_to_measure
    - max_age_years
    - min_age_years
    - study_type
    - model
    - primary_purpose
    - num_countries
    - agency_class

- Label:
    - terminated

- Highly sparse features were dropped (>95% missing).

- Missingness will be handled in the modeling pipeline.

In [27]:
final_table.head(10)

,nct_id,phase,is_fda_regulated_device,is_fda_regulated_drug,number_of_groups,number_of_arms,number_of_facilities,has_us_facility,number_of_primary_outcomes_to_measure,number_of_secondary_outcomes_to_measure,max_age_years,min_age_years,primary_purpose,study_type,model,num_countries,agency_class,terminated
0,NCT06790056,NaN,f,f,1.0,NaN,1,f,1.0,1.0,65.0,18.0,NaN,observational,COHORT,1.0,OTHER,False
1,NCT01256333,NaN,NaN,NaN,NaN,NaN,2,f,1.0,4.0,NaN,18.0,NaN,observational,CASE_ONLY,1.0,OTHER,False
2,NCT03980301,NaN,f,f,NaN,NaN,1,f,1.0,NaN,45.0,18.0,NaN,observational,COHORT,1.0,OTHER,False
3,NCT02689219,PHASE2,f,t,NaN,2.0,4,t,1.0,3.0,NaN,18.0,TREATMENT,intervention,PARALLEL,1.0,OTHER,True
4,NCT06101732,NaN,f,f,NaN,NaN,1,f,1.0,11.0,NaN,18.0,NaN,observational,COHORT,1.0,OTHER,False
5,NCT03816436,NaN,f,f,1.0,NaN,2,f,1.0,5.0,NaN,18.0,NaN,observational,COHORT,1.0,OTHER,False
6,NCT04578912,NaN,t,f,NaN,3.0,1,t,2.0,3.0,21.0,12.0,TREATMENT,intervention,PARALLEL,1.0,OTHER,False
7,NCT06796855,PHASE4,f,f,NaN,2.0,1,f,1.0,4.0,NaN,18.0,TREATMENT,intervention,PARALLEL,1.0,OTHER_GOV,False
8,NCT05602506,PHASE4,f,f,NaN,4.0,1,f,2.0,15.0,NaN,18.0,TREATMENT,intervention,PARALLEL,1.0,OTHER,False
9,NCT06115408,PHASE2/PHASE3,f,f,NaN,2.0,1,f,1.0,3.0,45.0,20.0,HEALTH_SERVICES_RESEARCH,intervention,PARALLEL,1.0,OTHER,False


In [28]:
final_table.to_csv('../data/processed/model_data.csv', index=False)